### Phase 3: Modelling and Analysis

In [8]:
# ============================================================
# PHASE 3 — MODELING PIPELINE
# Alpha Prediction using Rolling Walk-Forward Training
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, precision_score


In [9]:
# ============================================================
# CONFIGURATION
# ============================================================

FEATURE_DIR = Path("data/features")  # Folder containing individual ticker CSVs

TRAIN_DAYS = 84
PURGE_DAYS = 21
TEST_DAYS = 21
STEP_SIZE = 21

START_DATE = "2020-01-01"
TOP_N = 10


In [10]:
# ============================================================
# STEP 1: LOAD PANEL
# ============================================================

print("📥 Loading feature files...")

dfs = []

for fp in FEATURE_DIR.glob("*.csv"):

    df = pd.read_csv(fp)
    df.columns = df.columns.str.lower()
    
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    if "ticker" not in df.columns:
        df["ticker"] = fp.stem

    dfs.append(df)

panel = pd.concat(dfs, ignore_index=True)

panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(["ticker","date"]).reset_index(drop=True)

print("Panel rows:", len(panel))


📥 Loading feature files...
Panel rows: 131555


In [11]:
# ============================================================
# STEP 2: FEATURE LIST (STRICTLY AS PROVIDED)
# ============================================================

FEATURES = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "momentum_1m",
    "momentum_3m",
    "momentum_6m",
    "vol_90d",
    "rsi_14",
    "macd_hist",
    "concall_sentiment_score",
    "concall_keyword_intensity"
]


In [12]:
# ============================================================
# STEP 3: TARGET CREATION
# future 21-day return classification
# ============================================================

panel["future_return_21d"] = panel.groupby("ticker")["close"].shift(-21) / panel["close"] - 1
panel["target"] = (panel["future_return_21d"] > 0).astype(int)


In [13]:
# ============================================================
# STEP 4: CROSS-SECTIONAL Z-SCORE NORMALIZATION
# ============================================================

for col in FEATURES:
    panel[col] = panel.groupby("date")[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-9)
    )


KeyError: 'Column not found: concall_sentiment_score'

In [14]:
# ============================================================
# STEP 5: FILTER DATE RANGE
# ============================================================

panel = panel[panel["date"] >= START_DATE].copy()
unique_dates = sorted(panel["date"].unique())

# ============================================================
# STEP 6: MODEL
# ============================================================

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# ============================================================
# STEP 7: WALK-FORWARD TRAINING
# ============================================================

all_predictions = []
metrics = []

i = 0
while True:

    train_start = i
    train_end = train_start + TRAIN_DAYS
    purge_end = train_end + PURGE_DAYS
    test_end = purge_end + TEST_DAYS

    if test_end >= len(unique_dates):
        break

    train_dates = unique_dates[train_start:train_end]
    purge_dates = unique_dates[train_end:purge_end]
    test_dates = unique_dates[purge_end:test_end]

    train_df = panel[panel["date"].isin(train_dates)]
    test_df = panel[panel["date"].isin(test_dates)]

    X_train = train_df[FEATURES]
    y_train = train_df["target"]

    X_test = test_df[FEATURES]
    y_test = test_df["target"]

    model.fit(X_train, y_train)

    prob = model.predict_proba(X_test)[:, 1]

    test_df = test_df.copy()
    test_df["prediction"] = prob

    # =========================
    # METRICS
    # =========================

    auc = roc_auc_score(y_test, prob)
    pred_label = (prob > 0.5).astype(int)
    precision = precision_score(y_test, pred_label)

    metrics.append(
        {
            "train_start": train_dates[0],
            "train_end": train_dates[-1],
            "test_start": test_dates[0],
            "test_end": test_dates[-1],
            "AUC": auc,
            "Precision": precision
        }
    )

    all_predictions.append(test_df)

    i += STEP_SIZE

# ============================================================
# STEP 8: MERGE ALL TEST PREDICTIONS
# ============================================================

predictions_df = pd.concat(all_predictions).reset_index(drop=True)
metrics_df = pd.DataFrame(metrics)

# ============================================================
# STEP 9: FINAL TRAINING FOR LATEST MODEL
# Train = n-105 → n-21
# Purge = n-21 → n
# ============================================================

last_train_start = - (TRAIN_DAYS + PURGE_DAYS)
last_train_end = - PURGE_DAYS

train_dates = unique_dates[last_train_start:last_train_end]
train_df = panel[panel["date"].isin(train_dates)]

X_train = train_df[FEATURES]
y_train = train_df["target"]

model.fit(X_train, y_train)

# ============================================================
# STEP 10: RANK STOCKS FOR NEXT 21 TRADING DAYS
# ============================================================

latest_date = panel["date"].max()
latest_df = panel[panel["date"] == latest_date].copy()

latest_df["prediction"] = model.predict_proba(latest_df[FEATURES])[:, 1]
latest_df = latest_df.sort_values("prediction", ascending=False)
top_stocks = latest_df.head(TOP_N)

# ============================================================
# STEP 11: OUTPUT RESULTS
# ============================================================

print("\n==============================")
print("MODEL PERFORMANCE SUMMARY")
print("==============================")
print(metrics_df.describe())

print("\n==============================")
print("TOP 10 STOCK PICKS")
print("Predicted for next 21 trading days")
print("==============================")
print(top_stocks[["date", "ticker", "prediction"]])

# ============================================================
# STEP 12: SAVE RESULTS
# ============================================================

predictions_df.to_csv("phase3_predictions.csv", index=False)
metrics_df.to_csv("phase3_metrics.csv", index=False)
top_stocks.to_csv("phase3_top10_next_month.csv", index=False)

print("\nPhase 3 pipeline completed successfully.")

KeyError: "['concall_sentiment_score', 'concall_keyword_intensity'] not in index"

In [15]:
dfs

[     symbol series       date prevclose    open     high      low lastprice  \
 0       ABB     EQ 2020-07-09    944.65   947.0   955.00   926.05    928.50   
 1       ABB     EQ 2020-07-10    928.50   930.0   936.75   914.90    915.15   
 2       ABB     EQ 2020-07-13    916.95   922.0   925.35   910.00    913.00   
 3       ABB     EQ 2020-07-14    914.90   913.0   915.00   901.05    910.50   
 4       ABB     EQ 2020-07-15    908.15   916.0   933.00   900.90    906.10   
 ...     ...    ...        ...       ...     ...      ...      ...       ...   
 1400    ABB     EQ 2026-02-24  5,917.50  5881.5  6075.00  5881.50  6,060.00   
 1401    ABB     EQ 2026-02-25  6,056.00  6101.0  6180.00  6046.50  6,158.00   
 1402    ABB     EQ 2026-02-26  6,167.50  6170.0  6205.50  6110.50  6,135.00   
 1403    ABB     EQ 2026-02-27  6,133.00  6128.0  6128.00  6039.00  6,080.00   
 1404    ABB     EQ 2026-03-02  6,073.00  5840.0  6073.00  5835.00  5,980.00   
 
         close averageprice  ...  sent